# Error Analysis & Hyperparameter Tuning — Notes

Companion notes to `mushroom_classification.ipynb`. Example figures are from the
mushroom dataset (8,124 rows; 80/20 split → 1,625 test rows).

## 1. Error Analysis

**Definition:** error analysis is the practice of systematically examining a model's
mistakes — *where* and *why* it fails — to guide improvements rather than just
reporting accuracy.

**Why:** accuracy hides *where* a model fails. For mushrooms, the deadly error is a
poisonous mushroom called edible (a **false negative** on class `1`).

**Confusion matrix** (shallow tree, `max_depth=3`):

| | Pred. edible (0) | Pred. poisonous (1) |
|---|---|---|
| Actual edible (0) | TN — safe | FP — edible flagged (annoying) |
| Actual poisonous (1) | **FN — poison passed as edible (dangerous)** | TP — caught |

Result: TN=816, FP=26, FN=3, TP=780 → 29/1625 errors (98.2% acc). Only 3 were the
dangerous kind.

**Precision vs recall:** precision = of those flagged, how many really were;
recall = of all poisonous, how many caught. Low recall = dangerous misses. F1 =
harmonic mean of both.

**Inspect mistakes:** pull misclassified rows and look for common features — on
this data errors cluster near specific `odor` / `gill-color` / `ring-type` values (the
model's blind spot).

**Threshold dial:** labels come from `prob > threshold` (default 0.5). Lower → more
flags → higher recall, lower precision; raise → the reverse. On 5%-poisonous
simulated data: default 0.5 caught 0% of poisons; threshold 0.05 caught 87%
(at 12% precision).

## 2. Hyperparameter Tuning

**Definition:** a hyperparameter is a configuration setting chosen *before* training
starts (e.g. `max_depth`, `n_estimators`) — unlike the weights the algorithm learns
from data. Hyperparameter tuning is the systematic search for the settings that give
the best validation performance.

**Core tradeoff:** too simple → underfits; too complex → overfits (memorizes noise).
On a noisy checkerboard demo, test accuracy peaked at `max_depth=5` then fell.

**Golden rule:** tune with **cross-validation**; use the test set once, at the end.

**Validation curve:** plot CV train vs validation score vs a hyperparameter. On
mushrooms, validation F1 saturates near `max_depth=5` (0.9997); deeper trees only
raise the train score = overfitting.

**GridSearchCV** (exhaustive): tried all combos of `max_depth ∈ {3,5,7,10,None}` ×
`min_samples_leaf ∈ {1,2,5}` → best `{max_depth:5, min_samples_leaf:1}`, CV F1 0.9997.

**RandomizedSearchCV** (sampled): 30 random points → `{max_depth:6, min_samples_split:4,
min_samples_leaf:1, criterion:entropy}`, CV F1 0.9997 — same quality, far bigger space.
Use when a grid would explode.

**Final check (once):** test accuracy ≈ 0.9988, F1 ≈ 0.9987.

## Takeaways

1. Read the confusion matrix, not just accuracy.
2. Precision/recall trade off — set it for the problem (on mushrooms, favor recall).
3. Inspect misclassified rows to find blind spots.
4. The threshold is a dial — tune to the cost of errors.
5. Hyperparameters balance underfit ↔ overfit.
6. Tune with CV; test set once.
7. Grid = exhaustive; Random = scales to big spaces.

## Resources

- Grid search: https://scikit-learn.org/stable/modules/grid_search.html
- Metrics & confusion matrix: https://scikit-learn.org/stable/modules/model_evaluation.html#confusion-matrix
- Validation curve: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.validation_curve.html